In [16]:
# Software Qualification Report - RIWS Co-registration Accuracy
# Copyright 2026 B-Open
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Introduction and scope

The objective of this notebook is to provide the evidence of the qualification step of the COP ARD Processors for ROSE-L data in Interferometric Wide Swath (RIWS) mode.

In particular, this notebook presents the Azimuth Co-registration Accuracy and Burst Overlap Phase Discontinuity analysis, demonstrating the results of test case **TC-ST-PERF-008**, as defined in the Software Verification Plan (Reference: esa-rose-l-015-D15, Issue/Revision: 1.1).

### Input Products

The TC-ST-PERF-008 qualification dataset includes the following products:

1. **ROSE-L RIWS ARD CSLC Products (Geometric & ESD):**  
   Simulated ROSE-L RIWS dual-polarization (HH/HV) CSLC products generated by the `cop-ard-processor` under geometric-only and Enhanced Spectral Diversity (ESD) configurations.
2. **ROSE-L RIWS GPP Reference CSLC Product:**  
   `TD_ROSEL_GPP_RIWS` reference coregistered CSLC product generated by the Ground Prototype Processor (GPP), used as the baseline truth for along-track alignment and burst overlap phase continuity.

## TC-ST-PERF-008: Azimuth Co-registration Accuracy (Data-Driven ROSE-L)

#### CSLC Product Quality (Data-Driven ROSE-L)

<table class="table table-striped table-bordered" style="width:100%;">
  <thead>
    <tr>
      <th style="width: 22%;">Field</th>
      <th style="width: 78%;">Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Test Case ID</strong></td>
      <td>TC-ST-PERF-008</td>
    </tr>
    <tr>
      <td><strong>Requirement(s)</strong></td>
      <td>APP-PERF-009</td>
    </tr>
    <tr>
      <td><strong>Objective</strong></td>
      <td>
        To quantitatively verify that the APP Co-registration Unit (using Enhanced Spectral Diversity, ESD) achieves the required azimuth co-registration accuracy, using the coregistered CSLC from the Ground Prototype Processor (GPP) as reference. The test verifies that burst overlap phase discontinuities are corrected and the residual azimuth misregistration with respect to the GPP reference is $\le 2\%$ of the azimuth pixel spacing.
      </td>
    </tr>
    <tr>
      <td><strong>Test Data</strong></td>
      <td>
        <strong>TD_ROSEL_GPP_RIWS:</strong> Simulated ROSE-L RIWS mode L1 SLC dataset and the corresponding reference GPP coregistered CSLC product.
      </td>
    </tr>
    <tr>
      <td><strong>Pre-conditions</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>APP installed on CEM-PAL facility.</li>
          <li>A Local Dask cluster configured in single-machine, multi-process mode is active.</li>
          <li>The input L1 SLC dataset and the reference GPP coregistered CSLC products are available.</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Configuration</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li><strong>Polarization:</strong> HH + HV (or HH)</li>
          <li><strong>ETAD:</strong> No</li>
          <li><strong>ICC:</strong> No</li>
          <li><strong>Split-Spectrum:</strong> No</li>
          <li><strong>ESD:</strong> Yes</li>
          <li><strong>Logging Level:</strong> INFO</li>
          <li><strong>Checkpoints:</strong> DISABLE</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Procedure</strong></td>
      <td>
        Execution of <code>TC-ST-PERF-008</code> implementing:
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>Initialize the APP with a Job Order configured to process the simulated ROSE-L RIWS dataset.</li>
          <li>Execute the coregistration pipeline: Model-based geometric coregistration followed by Enhanced Spectral Diversity (ESD) to estimate and correct sub-pixel azimuth shifts.</li>
          <li>Generate the final coregistered CSLC product.</li>
          <li>Compute double-difference burst overlap interferograms to evaluate phase continuity across adjacent burst boundaries.</li>
          <li>Form a differential interferogram between the APP CSLC and the reference GPP CSLC to quantify residual along-track phase and shift discrepancies.</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Pass Criteria</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>The processor terminates with Exit Code 0.</li>
          <li>Burst overlap phase step discontinuities are eliminated, resulting in residual mean overlap phases centered at $\approx 0\text{ rad}$.</li>
          <li>The absolute residual azimuth co-registration discrepancy between the APP product and the GPP reference is $\le 2\%$ of the azimuth pixel spacing.</li>
        </ul>
      </td>
    </tr>
  </tbody>
</table>

In [2]:
import re
import xml.etree.ElementTree as ET
import dask.array as da
import os
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path
from cop_ard_app.computation import create_client, map
from cop_ard_app.io import read, write, glob, delete
from cop_ard_app.esd import estimate
from cop_ard_app.utils import split_slc_datatrees

In [3]:
# ---------------------------------------------------------------------------
# 1. PARSER XML ARESYS
# ---------------------------------------------------------------------------
def _parse_val(text: str):
    """Converte una stringa in float o int se possibile."""
    if text is None:
        return None
    text = text.strip()
    if not text:
        return ""
    try:
        if "." in text or "e" in text.lower():
            return float(text)
        return int(text)
    except ValueError:
        return text


def _elem_to_dict(elem: ET.Element):
    """Converte ricorsivamente un nodo XML in dizionario Python."""
    children = list(elem)
    # Se il nodo contiene una sequenza di <val N="...">
    if children and all(child.tag == "val" for child in children):
        try:
            sorted_children = sorted(children, key=lambda c: int(c.attrib.get("N", 0)))
        except ValueError:
            sorted_children = children
        return np.array([_parse_val(c.text) for c in sorted_children])

    if not children:
        return _parse_val(elem.text)

    node_data = {}
    for child in children:
        child_val = _elem_to_dict(child)
        if child.tag in node_data:
            if not isinstance(node_data[child.tag], list):
                node_data[child.tag] = [node_data[child.tag]]
            node_data[child.tag].append(child_val)
        else:
            node_data[child.tag] = child_val
    return node_data


def parse_aresys_xml(xml_path: Path) -> dict:
    """Legge e converte il file XML Aresys."""
    tree = ET.parse(xml_path)
    return _elem_to_dict(tree.getroot())


def _parse_utc_timestamp(utc_str: str) -> pd.Timestamp:
    """Converte date Aresys (es. '31-DEC-1999 23:59:58.077659182407') in Timestamp."""
    match = re.match(r"^(\d{2}-[A-Za-z]{3}-\d{4} \d{2}:\d{2}:\d{2})(\.\d+)?", str(utc_str))
    if match:
        base, frac = match.groups()
        frac = (frac or "")[:9]  # Fino a nanosecondi
        return pd.to_datetime(f"{base}{frac}", format="mixed")
    return pd.to_datetime(utc_str)


def _get_dtype(cell_type: str, byte_order: str) -> np.dtype:
    """Restituisce il tipo NumPy corrispondente al raster."""
    endian = "<" if str(byte_order).upper() == "LITTLEENDIAN" else ">"
    type_map = {
        "FLOAT_COMPLEX": f"{endian}c8",  # complex64
        "DOUBLE_COMPLEX": f"{endian}c16",  # complex128
        "FLOAT": f"{endian}f4",
        "DOUBLE": f"{endian}f8",
        "SHORT": f"{endian}i2",
        "INT": f"{endian}i4",
    }
    return np.dtype(type_map.get(cell_type, f"{endian}c8"))


# ---------------------------------------------------------------------------
# 2. GESTIONE DEI BURST E RAGGRUPPAMENTO POLARIZZAZIONI
# ---------------------------------------------------------------------------
def _compute_burst_slices(burst_info: dict, total_lines: int):
    """Calcola gli intervalli di linee [start, end) per ciascun burst."""
    num_bursts = int(burst_info.get("NumberOfBursts", 1))
    change_list = burst_info.get("LinesPerBurstChangeList", {})
    lines_cfg = change_list.get("Lines") if isinstance(change_list, dict) else None

    if isinstance(lines_cfg, (int, float)):
        lines_per_burst = [int(lines_cfg)] * num_bursts
    else:
        lines_per_burst = [total_lines // num_bursts] * num_bursts

    slices = []
    current_line = 0
    for l in lines_per_burst:
        slices.append((current_line, current_line + l))
        current_line += l
    return slices


def load_aresys_product_to_datatree(folder_path: str | Path) -> xr.DataTree:
    """Scansiona la cartella, raggruppa le polarizzazioni per Swath,

    suddivide i raster in burst e restituisce un DataTree.
    """
    folder = Path(folder_path)
    xml_files = sorted(folder.glob("iSLC_Cor_*.xml"))

    if not xml_files:
        raise FileNotFoundError(f"Nessun file XML trovato in {folder}")

    # Raggruppa i canali per Swath (es. SS1) e Polarizzazione (es. H/H, H/V)
    # Struttura: grouped_swaths[swath][polarization] = {'meta': ..., 'raster_path': ...}
    grouped_swaths = defaultdict(dict)

    for xml_path in xml_files:
        meta = parse_aresys_xml(xml_path)
        channel = meta.get("Channel")
        if isinstance(channel, list):
            channel = channel[0]

        swath = channel.get("SwathInfo", {}).get("Swath", "SWATH_0")
        pol = channel.get("SwathInfo", {}).get("Polarization", "RAW")

        raster_name = channel["RasterInfo"]["FileName"]
        raster_path = folder / raster_name

        if not raster_path.exists():
            print(f"Avviso: Raster {raster_path.name} non trovato, salto.")
            continue

        grouped_swaths[swath][pol] = {
            "meta": channel,
            "raster_path": raster_path,
        }

    tree_dict = {}

    # Itera su ciascuna Swath trovata
    for swath_name, pol_dict in grouped_swaths.items():
        pol_keys = sorted(pol_dict.keys())
        ref_channel = pol_dict[pol_keys[0]]["meta"]
        raster_info = ref_channel["RasterInfo"]
        burst_info = ref_channel["BurstInfo"]

        total_lines = int(raster_info["Lines"])
        samples = int(raster_info["Samples"])
        offset = int(raster_info.get("HeaderOffsetBytes", 0))
        lines_step = float(raster_info["LinesStep"])
        samples_step = float(raster_info["SamplesStep"])
        dtype = _get_dtype(raster_info["CellType"], raster_info["ByteOrder"])

        # Mappa in memoria i file raster di ciascuna polarizzazione
        mmap_pols = {}
        for pol in pol_keys:
            r_path = pol_dict[pol]["raster_path"]
            mmap_pols[pol] = np.memmap(
                r_path,
                dtype=dtype,
                mode="r",
                offset=offset,
                shape=(total_lines, samples),
            )

        # Calcola le fette di linee per burst
        burst_slices = _compute_burst_slices(burst_info, total_lines)
        burst_meta_list = burst_info.get("Burst", [])
        if isinstance(burst_meta_list, dict):
            burst_meta_list = [burst_meta_list]

        # Costruisce il Dataset per ciascun burst
        for b_idx, (b_start, b_end) in enumerate(burst_slices):
            burst_lines = b_end - b_start
            b_meta = burst_meta_list[b_idx] if b_idx < len(burst_meta_list) else {}

            # 1. Coordinate Slant Range Time (secondi)
            r_start = float(b_meta.get("RangeStartTime", raster_info["SamplesStart"]))
            slant_range_time = r_start + np.arange(samples) * samples_step

            # 2. Coordinate Azimuth Time (UTC datetime)
            az_start_raw = b_meta.get("AzimuthStartTime", raster_info["LinesStart"])
            az_start = _parse_utc_timestamp(str(az_start_raw))
            azimuth_time = pd.date_range(
                start=az_start,
                periods=burst_lines,
                freq=pd.to_timedelta(lines_step, unit="s"),
            )

            burst_slices_pol = [mmap_pols[pol][b_start:b_end, :] for pol in pol_keys]

            dask_slices = [da.from_array(s, chunks=(burst_lines, samples)) for s in burst_slices_pol]
            stacked_data = da.stack(dask_slices, axis=0)

            da_burst = xr.DataArray(
                stacked_data,
                dims=("polarization", "azimuth_time", "slant_range_time"),
                coords={
                    "polarization": pol_keys,
                    "azimuth_time": azimuth_time,
                    "slant_range_time": slant_range_time,
                },
                attrs={
                    "burst_number": b_idx + 1,
                    "lines_step_s": lines_step,
                    "samples_step_s": samples_step,
                    "burst_center_azimuth_shift_s": float(b_meta.get("BurstCenterAzimuthShift", 0.0)),
                },
            )

            ds_burst = xr.Dataset(
                data_vars={"slc": da_burst},
                attrs={
                    "swath": swath_name,
                    "burst_id": b_idx + 1,
                    "prf_hz": float(ref_channel.get("SwathInfo", {}).get("AcquisitionPRF", 0)),
                    "radar_frequency_hz": float(ref_channel.get("DataSetInfo", {}).get("fc_hz", 0)),
                },
            )

            node_path = f"/{swath_name}/burst_{b_idx + 1:02d}"
            tree_dict[node_path] = ds_burst

    return xr.DataTree.from_dict(tree_dict)


def read_burst(path_burst):
    name = path_burst[-9:]
    cslc = read(f"{path_burst}/measurements", as_type="xr.Dataset", consolidated=False).drop_attrs()
    return (name, cslc)


def concat(name, bursts):
    sw, id = name.split("_")
    n = f"ss{sw[1]}_{int(id)}"
    ref_burst = read(f"{reference_path}/bursts/{n}/reference/measurements", consolidated=False).slc
    attrs = ref_burst.attrs
    azimuth_m = ref_burst.azimuth_m.values
    slant_range_m = ref_burst.slant_range_m.values
    c = (
        xr.concat(bursts, dim="acquisition_id")
        .assign_coords({"azimuth_m": ("azimuth_time", azimuth_m), "slant_range_m": ("slant_range_time", slant_range_m)})
        .assign_attrs(attrs)
    ).sortby("acquisition_date")
    return (n, c)

In [4]:
folder = "/scratch/external/COP-ARD/ESD-ROSEL/sarint/iSLC_Cor"
dt = load_aresys_product_to_datatree(folder)

In [5]:
multiburst = True
cslc_path = "s3://bopen-sar/ARD-ROSE-L/GPP_V2.1/OUTPUT/TC-ST-PERF-008/CSLC"
reference_path = "s3://bopen-sar/ARD-ROSE-L/GPP_V2.1/WDIR/TC-ST-PERF-008/REFDB/reference_db_static"

In [6]:
ref_file = f"{cslc_path}/cslcs_1999-12-31.zarr"
mode_to_file = {"esd": f"{cslc_path}/cslcs_2000-01-01-esd.zarr", "geom": f"{cslc_path}/cslcs_2000-01-01.zarr"}

In [ ]:
data = {}
data["gpp"] = dt
for mode, file in mode_to_file.items():
    files = [ref_file, file]
    acquisition_dates = sorted([k.split("cslcs_")[1].removesuffix(".zarr") for k in files])
    ref_path = files[0]
    for counter, sec_path in enumerate(files[1:]):
        print("=" * 50)

        ref_date = acquisition_dates[counter]
        sec_date = acquisition_dates[counter + 1]
        if ref_date not in ref_path:
            raise ValueError(f"Reference date {ref_date} not found in {ref_path}")
        if sec_date not in sec_path:
            raise ValueError(f"Secondary date {sec_date} not found in {sec_path}")
        print(f"Computing: {ref_date}  <--->  {sec_date}")

        bursts_dict = defaultdict(list)

        for f in [ref_path, sec_path]:
            bursts = glob(f"{f}/*CSL*")
            res = map(read_burst, bursts)
            for k, v in res:
                bursts_dict[k].append(v)

        args = [list(bursts_dict.keys()), list(bursts_dict.values())]
        res = map(concat, args, enable_dask=True)
        cslcs = xr.DataTree()
        for k, v in res:
            cslcs[f"bursts/{k}/coreg/measurements"] = v
            cslcs[f"bursts/{k}/orig"] = xr.Dataset()
    data[mode] = cslcs

In [8]:
def time_to_pixel_az(stack: xr.Dataset, dtaz: float, reference_time: None = None):
    if reference_time is None:
        reference_time = stack.azimuth_time[0].values
    azimuth_pixel = np.round((stack.azimuth_time - reference_time).astype("float64") / 1e9 / dtaz).astype("int32")
    stack = stack.assign_coords({"azimuth_pixel": azimuth_pixel})

    return stack.swap_dims(azimuth_time="azimuth_pixel"), reference_time

In [91]:
def compute_phase(cslcs, current_id, reference_time, gpp_data=None):
    ml = {"azimuth_pixel": 16, "slant_range_time": 8}
    s, id = current_id.split("_")
    next_id = f"{s}_{int(id) + 1}"

    cstack_next = cslcs.bursts[next_id].coreg.measurements.cslc.sel(polarization="HH")
    phase_model_next = cslcs.bursts[next_id].coreg.measurements.phase_model
    cstack_next = (cstack_next * np.exp(-1j * phase_model_next)).compute()

    cstack = cslcs.bursts[current_id].coreg.measurements.cslc.sel(polarization="HH")
    phase_model = cslcs.bursts[current_id].coreg.measurements.phase_model
    cstack = (cstack * np.exp(-1j * phase_model)).compute()

    annot = cslcs.bursts[current_id].coreg.measurements.attrs
    pol = "HH"
    annot = annot[pol]
    dtaz = annot["RMA"]["INCA"]["TimeCAPoly"]["Coefs"][1] * annot["Grid"]["Col"]["SS"]
    current_burst_ref, _ = time_to_pixel_az(cstack[0], dtaz, reference_time)
    current_burst_sec, _ = time_to_pixel_az(cstack[1], dtaz, reference_time)
    next_burst_ref, _ = time_to_pixel_az(cstack_next[0], dtaz, reference_time)
    next_burst_sec, _ = time_to_pixel_az(cstack_next[1], dtaz, reference_time)

    if gpp_data is not None:
        ci = gpp_data[s.upper()][f"burst_0{int(id)}"].slc.sel(polarization="H/H")
        cn = gpp_data[s.upper()][f"burst_0{int(id) + 1}"].slc.sel(polarization="H/H")

        current_burst_sec[:] = ci.values
        next_burst_sec[:] = cn.values

    intf1 = current_burst_sec * current_burst_ref.conj()
    intf2 = next_burst_sec * next_burst_ref.conj()
    intf = intf2 * intf1.conj()
    intf = intf.coarsen(ml, boundary="trim").mean().mean(dim="slant_range_time")
    phase = xr.ufuncs.angle(intf).compute()
    phase = phase.where(abs(intf) != 0, np.nan)

    return phase


def compute_shift(data, overlap_phase):
    burst0 = data["geom"].bursts[k].coreg.measurements.to_dataset()
    burst1 = data["geom"].bursts[next_id].coreg.measurements.to_dataset()
    annot0 = burst0.attrs["HH"]
    coeffs0 = annot0["Grid"]["Col"]["DeltaKCOAPoly"]["Coefs"]
    dazm0 = annot0["Grid"]["Col"]["SS"]

    annot1 = burst1.attrs["HH"]
    coeffs1 = annot1["Grid"]["Col"]["DeltaKCOAPoly"]["Coefs"]
    dazm1 = annot1["Grid"]["Col"]["SS"]

    dazm = (dazm0 + dazm1) / 2.0

    dtaz = annot0["RMA"]["INCA"]["TimeCAPoly"]["Coefs"][1] * annot0["Grid"]["Col"]["SS"]
    burst0, _ = time_to_pixel_az(burst0, dtaz, reference_time)
    burst1, _ = time_to_pixel_az(burst1, dtaz, reference_time)
    sel = burst0.drop_vars(burst0.data_vars) + burst1.drop_vars(burst1.data_vars)

    burst0 = burst0.sel({k: sel.coords[k] for k in ["azimuth_pixel", "slant_range_time"]})
    burst1 = burst1.sel({k: sel.coords[k] for k in ["azimuth_pixel", "slant_range_time"]})

    # 3. Evaluate both polynomials at the exact same physical coordinates
    sm0, am0 = np.meshgrid(burst0.slant_range_m, burst0.azimuth_m)
    sm1, am1 = np.meshgrid(burst1.slant_range_m, burst1.azimuth_m)

    # doppler centroid in the area of interest
    f_d_0 = np.polynomial.polynomial.polyval2d(sm0, am0, coeffs0)
    f_d_1 = np.polynomial.polynomial.polyval2d(sm1, am1, coeffs1)

    # 4. Compute spectral separation and wrap cycle
    delta_k = np.abs(np.mean(f_d_1 - f_d_0))  # [cycles / meter]
    wrap_cycle = 1.0 / (delta_k * dazm)  # [pixels / cycle]

    # 5. Convert overlap phase to residual azimuth pixel shift
    diff_shift = (overlap_phase["esd"] - overlap_phase["gpp"]) / (2.0 * np.pi) * wrap_cycle
    return diff_shift

In [97]:
profiles = {}
shifts = {}
product = next(iter(data["geom"].bursts.values())).coreg.measurements.cslc
reference_time = product.azimuth_time[0].values

burst_keys = sorted(cslcs.bursts.keys(), key=lambda x: (x.split("_")[0], int(x.split("_")[1])))

for k in burst_keys:
    s, id_str = k.split("_")
    next_id = f"{s}_{int(id_str) + 1}"
    if next_id not in cslcs.bursts:
        continue

    # Structured key: (subswath, interface) e.g., ("ss1", "1 -> 2")
    pair_label = f"{s} ({id_str} -> {int(id_str) + 1})"

    profiles[pair_label] = {
        "geom": compute_phase(data["geom"], k, reference_time),
        "esd": compute_phase(data["esd"], k, reference_time),
        "gpp": compute_phase(data["geom"], k, reference_time, data["gpp"]),
    }
    shifts[pair_label] = compute_shift(data, profiles[pair_label])

In [98]:
subswaths = ["ss1", "ss2", "ss3"]
interfaces = ["1 -> 2", "2 -> 3", "3 -> 4"]

fig, axes = plt.subplots(
    len(subswaths),
    len(interfaces),
    figsize=(14, 7.0),
    sharey=True,
    sharex=False,  # Swaths have different overlap lengths; let each row fit edge-to-edge
    constrained_layout=True,
)

for r, ss in enumerate(subswaths):
    for c, pair in enumerate(interfaces):
        ax = axes[r, c]
        key = f"{ss} ({pair})"

        if key not in profiles:
            ax.set_visible(False)
            continue

        p = profiles[key]
        g_vals = p["geom"].values
        e_vals = p["esd"].values
        r_vals = p["gpp"].values

        # Reset coordinate to 0..N so lines stretch across the full subplot width
        x_rel = p["geom"].azimuth_pixel.values

        (l1,) = ax.plot(x_rel, g_vals, color="#c0392b", lw=1.2, label="Geometric")
        (l2,) = ax.plot(x_rel, e_vals, color="#27ae60", lw=2.2, alpha=0.6, label="ESD (ARD)")
        (l3,) = ax.plot(
            x_rel,
            r_vals,
            color="#2980b9",
            lw=1.1,
            linestyle="--",
            label="GPP (Reference)",
        )

        ax.axhline(0, color="gray", linestyle=":", lw=0.8)
        ax.set_ylim(-0.35, 0.1)
        # ax.set_xlim(0, len(x_rel) - 1)  # No empty whitespace margins
        ax.grid(True, linestyle="--", alpha=0.3)

        # Interface column headers (Top row only)
        if r == 0:
            ax.set_title(f"Interface {pair}", fontsize=11, fontweight="bold", pad=8)

        # Subswath labels directly on the LEFT Y-axis
        if c == 0:
            ax.set_ylabel(
                f"{ss.upper()}\nPhase [rad]",
                fontsize=11,
                fontweight="bold",
                labelpad=10,
            )

        # X-label only on bottom row
        if r == len(subswaths) - 1:
            ax.set_xlabel("Azimuth Pixel", fontsize=10)

fig.legend(
    [l1, l2, l3],
    ["Geometric", "ESD (COP-ARD-Processor)", "GPP (Reference)"],
    loc="upper center",
    bbox_to_anchor=(0.5, 1.05),
    ncol=3,
    frameon=False,
    fontsize=11,
)

fig.savefig("figures/riws_phase_interface.svg", format="svg", bbox_inches="tight")

plt.close(fig)

In [99]:
profiles = {}
ml = {"azimuth_time": 16, "slant_range_time": 8}
burst_keys = sorted(cslcs.bursts.keys(), key=lambda x: (x.split("_")[0], int(x.split("_")[1])))

for k in burst_keys:
    s, id_str = k.split("_")
    # Structured key: (subswath, interface)
    pair_label = f"{s} ({id_str})"

    cslc_cop = data["esd"].bursts[k].coreg.measurements.cslc.sel(polarization="HH").isel(acquisition_id=1)
    phase_model = data["esd"].bursts[k].coreg.measurements.phase_model.isel(acquisition_id=1)
    cslc_cop = (cslc_cop * np.exp(-1j * phase_model)).compute()
    cslc_gpp = data["gpp"][s.upper()][f"burst_0{id_str}"].slc.sel(polarization="H/H").values

    intf = (cslc_cop * cslc_gpp.conj()).coarsen(ml, boundary="trim").mean().compute()

    phase = xr.ufuncs.angle(intf).compute()
    phase = phase.where(abs(intf) != 0, np.nan)
    profiles[pair_label] = phase

In [100]:
from matplotlib.lines import Line2D

subswaths = ["ss1", "ss2", "ss3"]
burst_ids = ["1", "2", "3", "4"]

burst_colors = {
    "1": "#1f77b4",  # Blue
    "2": "#ff7f0e",  # Orange
    "3": "#2ca02c",  # Green
    "4": "#d62728",  # Red
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True, constrained_layout=True)

# Bin edges focused on the active distribution
bin_edges = np.linspace(-0.04, 0.02, 100)

for ax, ss in zip(axes, subswaths):
    for b_id in burst_ids:
        key = f"{ss} ({b_id})"
        if key not in profiles:
            continue

        vals = profiles[key].values.ravel()
        vals = vals[~np.isnan(vals)]

        # 1. Step outline
        ax.hist(
            vals,
            bins=bin_edges,
            density=True,
            histtype="step",
            color=burst_colors[b_id],
            lw=1.4,
            zorder=2,
        )

        # 2. Layered transparent fill
        ax.hist(
            vals,
            bins=bin_edges,
            density=True,
            histtype="stepfilled",
            color=burst_colors[b_id],
            alpha=0.08,
            zorder=1,
        )

    ax.set_title(f"Subswath {ss.upper()}", fontsize=11, fontweight="bold")
    ax.set_xlabel(r"$\phi_{\mathrm{ESD}} - \phi_{\mathrm{GPP}}$ [rad]", fontsize=10)
    ax.set_xlim(-0.04, 0.02)
    ax.grid(True, linestyle="--", alpha=0.3)

axes[0].set_ylabel("Probability Density", fontsize=10)

# Clean solid line handles (replaces the hollow [_] brackets)
legend_handles = [Line2D([0], [0], color=burst_colors[b], lw=2.0, label=f"Burst {b}") for b in burst_ids]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.07),
    ncol=4,
    frameon=False,
    fontsize=10,
)

fig.savefig("figures/riws_phase_difference.svg", format="svg", bbox_inches="tight")

plt.close(fig)


![Burst Overlap Phase Residuals](figures/riws_phase_interface.svg)  
*Figure 1: Burst overlap phase profiles across three subswaths (SS1–SS3, rows) and sequential burst interfaces (columns), averaged across the slant range time dimension.*

This figure compares the azimuth phase across adjacent burst overlap regions for three coregistration methods. For each interface, the 2D overlap interferogram was averaged across the slant range time dimension to obtain a clean 1D along-track profile:

- **Geometric (solid red):** Applying only geometric coregistration leaves a clear residual phase offset (ranging between $-0.15\text{ rad}$ and $-0.3\text{ rad}$) across all burst interfaces.
- **ESD (solid green):** Applying the Enhanced Spectral Diversity (ESD) correction from the `cop-ard-processor` eliminates the offset, centering the overlap phase at $\sim 0\text{ rad}$.
- **GPP (dashed blue):** The reference result from the Ground Prototype Processor closely tracks the ESD correction, also centering at $\sim 0\text{ rad}$.

Overall, the ESD correction successfully resolves the geometric phase jump, bringing the phase to $\sim 0\text{ rad}$ in tight alignment with the reference GPP processor.


![Phase Difference Distribution](figures/riws_phase_difference.svg)  
*Figure 2: Probability density distributions of the phase difference ($\phi_{\text{ESD}} - \phi_{\text{GPP}}$) between the `cop-ard-processor` and the GPP reference across subswaths SS1, SS2, and SS3.*

This figure illustrates the pixel-wise phase agreement between the ESD-corrected outputs and the reference Ground Prototype Processor (GPP):

- **Tight Alignment:** Across all three subswaths (SS1–SS3) and individual bursts (Bursts 1–4), the phase differences are highly concentrated near $0\text{ rad}$.
- **Consistency:** The probability density distributions are highly consistent across different bursts and subswaths, confirming robust phase agreement between the  processor and the reference implementation.


In [110]:
import pandas as pd

qc_summary = []

for pair_label, shift_da in shifts.items():
    vals = shift_da.values.ravel()
    vals = vals[~np.isnan(vals)]

    max_err = np.max(np.abs(vals))
    mean_err = np.mean(vals)

    qc_summary.append(
        {
            "Subswath / Interface": rf"{pair_label}",
            "Mean Bias [%]": f"{mean_err * 100:+.3f}%",
            "Max Abs Error [%]": f"{max_err * 100:.3f}%",
            "Pass (<= 2%)": max_err <= 0.02,
        }
    )

df_qc = pd.DataFrame(qc_summary)
df_qc

,Subswath / Interface,Mean Bias [%],Max Abs Error [%],Pass (<= 2%)
0,ss1 (1 -> 2),-0.109%,0.157%,True
1,ss1 (2 -> 3),-0.090%,0.124%,True
2,ss1 (3 -> 4),-0.073%,0.123%,True
3,ss2 (1 -> 2),+0.064%,0.113%,True
4,ss2 (2 -> 3),+0.042%,0.084%,True
5,ss2 (3 -> 4),+0.022%,0.095%,True
6,ss3 (1 -> 2),+0.030%,0.109%,True
7,ss3 (2 -> 3),+0.048%,0.119%,True
8,ss3 (3 -> 4),+0.027%,0.114%,True


## Results and Discussion

The Azimuth Co-registration analysis evaluates the geometric accuracy and along-track phase consistency of the CSLC products generated by the `cop-ard-processor` against the reference Ground Prototype Processor (GPP) for the ROSE-L RIWS acquisition mode.

### TC-ST-PERF-008 (Data-Driven / ESD)
- **Performance:** The Enhanced Spectral Diversity (ESD) correction successfully aligns the bursts across all three subswaths (`ss1`, `ss2`, `ss3`), eliminating burst overlap phase steps. The residual azimuth displacement between the processor output and the GPP reference is tightly constrained across all 9 burst interfaces, with mean along-track biases ranging from $-0.109\%$ to $+0.064\%$ of a pixel. The worst-case absolute error across the entire track is only $0.157\%$ of the azimuth pixel spacing (observed at interface `ss1 (1 -> 2)`).
- **Compliance:** All evaluated interfaces satisfy the qualification criteria with a safety margin exceeding an order of magnitude. The maximum measured absolute displacement ($0.157\%$) remains well within the mandated threshold of $\le 2.0\%$ of the azimuth pixel spacing, confirming full compliance with requirement **APP-PERF-009**.

<table class="table table-bordered table-hover" style="width: 100%; text-align: center; vertical-align: middle; margin-top: 15px;">
  <thead>
    <tr>
      <th style="text-align: left; width: 18%;">Test Case ID</th>
      <th style="width: 20%;">Processing Level</th>
      <th style="width: 22%;">Metric Category</th>
      <th style="width: 16%;">Target Threshold</th>
      <th style="width: 14%;">Worst Measured</th>
      <th style="width: 10%;">Verdict</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>TC-ST-PERF-008</strong></td>
      <td>Data-Driven (ESD)</td>
      <td>Azimuth Co-registration Accuracy</td>
      <td>&le; 2.0% px</td>
      <td>0.157% px</td>
      <td><span style="background-color: #28a745; color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold; font-size: 0.85em;">PASSED</span></td>
    </tr>
  </tbody>
</table>

#### Final Conclusion
Test case **TC-ST-PERF-008** successfully fulfills all azimuth co-registration accuracy criteria specified in the Software Verification Plan (D15v1.1). The `cop-ard-processor` CSLC processing chain is formally verified and qualified for ROSE-L RIWS mode product generation.